# Comprehensive Continued Pre-training (CPT) Tutorial

This notebook covers **continued pre-training (CPT)** — also called *continual* or
*domain-adaptive* pre-training — with the
[`training_hub`](https://github.com/Red-Hat-AI-Innovation-Team/training_hub) library on
Alauda AI.

CPT keeps the original **next-token prediction** objective but runs it over a **raw-text
corpus** from your domain (medical notes, legal contracts, code, a new language, ...). It
teaches the model new *knowledge and vocabulary* before any instruction tuning, unlike SFT
which teaches *behavior* from chat data.

`training_hub` runs CPT through the same `sft(...)` entrypoint with **`is_pretraining=True`**:
the loss is computed over *all* tokens (no assistant-only masking) and documents are packed
into fixed-size `block_size` windows.

## When to use CPT

A typical domain-adaptation pipeline is **CPT -> SFT -> alignment**:

1. **CPT** on raw domain text — inject knowledge / vocabulary.
2. **SFT** (or **OSFT**) on chat data — teach the model to follow instructions.
3. Optional preference/RL alignment.

| | CPT | SFT | OSFT |
|---|---|---|---|
| Data | raw text | chat (instruction/response) | chat |
| Objective | next-token over all tokens | next-token over assistant turns | next-token, orthogonal subspace |
| Teaches | knowledge / vocabulary | behavior | behavior, forgetting-resistant |

CPT can cause **catastrophic forgetting** of general ability. Mitigate by mixing in some
general-domain text, keeping the learning rate low, and running a follow-up SFT pass. If
forgetting is the main concern, consider **OSFT** instead (see
[Training Hub fine-tuning](../training-hub-fine-tuning.mdx)).

In [ ]:
import json
import os

# CPT trains on RAW TEXT: one document per line under a "text" field.
data_path = "./test_cpt_data.jsonl"
if not os.path.exists(data_path):
    print(f"Creating dummy raw-text corpus at {data_path}")
    docs = [
        "Alauda AI is an MLOps platform that runs fine-tuning and inference workloads on Kubernetes.",
        "Continued pre-training adapts a base language model to a new domain using unlabeled text.",
        "The training_hub library wraps SFT, OSFT, LoRA, QLoRA and continued pre-training behind one interface.",
        "Kubeflow Trainer v2 submits distributed TrainJobs that schedule onto GPU or NPU nodes.",
    ] * 8
    with open(data_path, "w") as f:
        for d in docs:
            f.write(json.dumps({"text": d}) + "\n")

## Data format requirements

CPT data is **raw text**, not chat turns. Each JSONL line is a document under a single
text column (default `"text"`):

```json
{"text": "A paragraph or document of raw domain text ..."}
{"text": "Another document ..."}
```

`training_hub` concatenates and chunks the documents into fixed-length `block_size`
windows and trains next-token prediction over every token (full unmasking).

- `document_column_name` — the JSONL field holding the text (default `"text"`).
- `block_size` — the packed context-window length (e.g. 1024-4096).

## Model configuration examples

CPT typically starts from a **base** (not already instruction-tuned) checkpoint. Provide a
HuggingFace name or a local path; pre-download to a local directory on air-gapped clusters.

In [ ]:
# =============================================================================
# MODEL CONFIGURATION EXAMPLES
# =============================================================================

# Example 1: Qwen 3 0.6B Base (tiny - good for a smoke test)
qwen_small_example = {
    "model_name": "Qwen 3 0.6B Base",
    "model_path": "/opt/app-root/src/Qwen3-0.6B-Base",
    "example_block_size": 1024,
    "example_batch_size": 8,
    "example_learning_rate": 5e-6,
}

# Example 2: Qwen 2.5 7B (base) - domain adaptation target
qwen_7b_example = {
    "model_name": "Qwen 2.5 7B",
    "model_path": "Qwen/Qwen2.5-7B",
    "example_block_size": 4096,
    "example_batch_size": 128,
    "example_learning_rate": 5e-6,
}

# =============================================================================
# SELECT YOUR CONFIGURATION
# =============================================================================
selected_example = qwen_small_example
print(f"Selected model: {selected_example['model_name']} ({selected_example['model_path']})")

## CPT parameter reference

CPT reuses every `sft(...)` parameter and adds three pre-training controls:

- `is_pretraining=True` — switch the data path to raw-text pre-training (no chat masking).
- `block_size` — packed sequence length for next-token prediction.
- `document_column_name` — JSONL field holding the raw text.

Use a **low learning rate** (CPT is sensitive — `1e-6`-`1e-5`) to limit forgetting.

In [ ]:
# =============================================================================
# COMPLETE CPT PARAMETER CONFIGURATION
# =============================================================================
from datetime import datetime

experiment_name = "cpt_comprehensive_example"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
full_experiment_name = f"{experiment_name}_{timestamp}"

# ---- Required ----
model_path = selected_example["model_path"]
data_path = "./test_cpt_data.jsonl"
ckpt_output_dir = f"checkpoints/{full_experiment_name}"

# ---- Continued pre-training controls ----
is_pretraining = True
block_size = selected_example["example_block_size"]
document_column_name = "text"

# ---- Core training ----
num_epochs = 1
effective_batch_size = selected_example["example_batch_size"]
learning_rate = selected_example["example_learning_rate"]   # keep low to limit forgetting
max_seq_len = block_size
max_tokens_per_gpu = 4096   # per-GPU token budget; auto micro-batch sizing
warmup_steps = 10

print("CPT configuration:")
print(f"  pretraining: is_pretraining={is_pretraining}, block_size={block_size}, column='{document_column_name}'")
print(f"  train: epochs={num_epochs}, ebs={effective_batch_size}, lr={learning_rate}, max_tokens_per_gpu={max_tokens_per_gpu}")

## Distributed configuration

CPT uses the same distributed topology knobs as SFT. Raw-text corpora are usually large,
so multi-GPU / multi-node is common — set `nproc_per_node` to the GPU count per node.

In [ ]:
nproc_per_node = 1   # raise to the per-node GPU count for real corpora
nnodes = 1
node_rank = 0
rdzv_id = 200
rdzv_endpoint = "127.0.0.1:29500"
print(f"distributed: nproc_per_node={nproc_per_node}, nnodes={nnodes}")

## Execute training

The final cell calls `training_hub.sft(..., is_pretraining=True)` — continued pre-training
over the raw-text corpus.

In [ ]:
from training_hub import sft

result = sft(
    # --- required ---
    model_path=model_path,
    data_path=data_path,
    ckpt_output_dir=ckpt_output_dir,
    # --- continued pre-training ---
    is_pretraining=is_pretraining,
    block_size=block_size,
    document_column_name=document_column_name,
    # --- core training ---
    num_epochs=num_epochs,
    effective_batch_size=effective_batch_size,
    learning_rate=learning_rate,
    max_seq_len=max_seq_len,
    max_tokens_per_gpu=max_tokens_per_gpu,
    warmup_steps=warmup_steps,
    checkpoint_at_epoch=True,
    # --- distributed ---
    nproc_per_node=nproc_per_node,
    nnodes=nnodes,
    node_rank=node_rank,
    rdzv_id=rdzv_id,
    rdzv_endpoint=rdzv_endpoint,
)
print(f"CPT finished: {result!r}")

## Post-training analysis

CPT writes a **full checkpoint** (the whole model is updated). The natural next step is an
SFT / OSFT pass on instruction data so the domain-adapted model can follow instructions.

In [ ]:
import os

hf_dir = os.path.join(ckpt_output_dir, "hf_format")
print(f"Checkpoint dir: {hf_dir}")
if os.path.isdir(hf_dir):
    for ckpt in sorted(os.listdir(hf_dir)):
        print("  checkpoint:", ckpt)

print("\nNext step: run sft() / osft() on instruction data, pointing model_path at this checkpoint.")

## Parameter reference summary

### CPT-specific parameters

| Parameter | Description | Typical |
|---|---|---|
| `is_pretraining` | Switch to raw-text pre-training (no chat masking) | `True` |
| `block_size` | Packed context-window length | 1024-4096 |
| `document_column_name` | JSONL field with raw text | `"text"` |

### Common training parameters

| Parameter | Description |
|---|---|
| `model_path`, `data_path`, `ckpt_output_dir` | Required I/O |
| `num_epochs`, `effective_batch_size`, `learning_rate` | Core training (use a **low** LR) |
| `max_seq_len`, `max_tokens_per_gpu` | Sequence length / memory budget |
| `nproc_per_node`, `nnodes`, `node_rank`, `rdzv_id`, `rdzv_endpoint` | Distributed topology |

> **NPU note:** Full-parameter continued pre-training also runs on Huawei Ascend NPU via
> the `MindSpeed-LLM` runtime (`pretrain_gpt.py`). See
> [Fine-tune and Pretrain on Ascend NPU](../fine-tune-and-pretrain-llms-on-ascend-npu.mdx)
> and the `qwen25_pretrain_verify.ipynb` recipe for the Megatron-style pre-training path.